In [1]:
# Remove conflicting versions
!pip uninstall -y gradio gradio-client websockets

# Install required libraries
!pip install gradio==3.39.0
!pip install gradio-client==0.3.0
!pip install numpy
!pip install scipy
!pip install matplotlib
!pip install gTTS
!pip install SpeechRecognition
!pip install pydub
!pip install transformers accelerate

Found existing installation: gradio 3.39.0
Uninstalling gradio-3.39.0:
  Successfully uninstalled gradio-3.39.0
Found existing installation: gradio_client 2.5.0
Uninstalling gradio_client-2.5.0:
  Successfully uninstalled gradio_client-2.5.0
Found existing installation: websockets 11.0.3
Uninstalling websockets-11.0.3:
  Successfully uninstalled websockets-11.0.3
  Using cached gradio-3.39.0-py3-none-any.whl.metadata (17 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached websockets-11.0.3-py3-none-any.whl.metadata (6.6 kB)
Using cached gradio-3.39.0-py3-none-any.whl (19.9 MB)
Using cached gradio_client-2.5.0-py3-none-any.whl (59 kB)
Using cached websockets-11.0.3-py3-none-any.whl (118 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires websockets>=14.0, but you have websockets 11.0.3 wh

In [2]:
import json
import math
import os
import random
import heapq
import re
import sqlite3

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label, distance_transform_edt
from gtts import gTTS
from pydub import AudioSegment
import speech_recognition as sr
# ============================================================
# DATABASE (UPDATED with special_needs column)
# ============================================================
DB_PATH = "mall.db"
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

# Create table with special_needs column (fresh start)
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    username TEXT PRIMARY KEY,
    password TEXT NOT NULL,
    name TEXT,
    age INTEGER,
    gender TEXT,
    lower_budget REAL,
    upper_budget REAL,
    special_needs INTEGER DEFAULT 0
)
""")

# Migrate old database: add special_needs if missing
try:
    cursor.execute("SELECT special_needs FROM users LIMIT 1")
except sqlite3.OperationalError:
    cursor.execute("ALTER TABLE users ADD COLUMN special_needs INTEGER DEFAULT 0")
    conn.commit()
    print("✅ Migrated database: added special_needs column")

cursor.execute("""
CREATE TABLE IF NOT EXISTS user_preferences (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT,
    main_category TEXT,
    category TEXT,
    subcategory TEXT,
    FOREIGN KEY(username) REFERENCES users(username)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT,
    start_point TEXT,
    end_store TEXT
)
""")
conn.commit()

In [3]:
cursor.execute("SELECT * FROM users")
print(cursor.fetchall())

cursor.execute("SELECT * FROM user_preferences")
print(cursor.fetchall())

cursor.execute("SELECT * FROM history")
print(cursor.fetchall())

[]
[]
[]


In [9]:
CATEGORIES_FILE = "/content/categories.json"
PRODUCTS_FILE = "/content/sort_data.json"

# ============================================================
# FAST LOCAL LLM FALLBACK (SmolLM2-135M-Instruct)
# ============================================================
try:
    from transformers import pipeline
    _llm_pipe = None

    def _get_llm():
        global _llm_pipe
        if _llm_pipe is None:
            print("⏳ Loading fast local model (SmolLM2-135M-Instruct)...")
            _llm_pipe = pipeline(
                "text-generation",
                model="HuggingFaceTB/SmolLM2-135M-Instruct",
                max_new_tokens=80,
                do_sample=False,
                return_full_text=False,
            )
            print("✅ Local LLM ready.")
        return _llm_pipe
except Exception:
    _get_llm = None


def _generate_llm_reply(user_text, lang="en"):
    if _get_llm is None:
        return None
    try:
        llm = _get_llm()
        if lang == "ar":
            prompt = (
                f"<|im_start|>user\nأنت مساعد ذكي في مول تجاري. رد باختصار جداً على: {user_text}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
        else:
            prompt = (
                f"<|im_start|>user\nYou are a smart mall assistant. Reply very briefly to: {user_text}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
        out = llm(prompt)[0]["generated_text"].strip()
        out = out.split("<|im_end|>")[0].split("<|im_start|>")[0].strip()
        return out if len(out) > 5 else None
    except Exception:
        return None




# Smart Mall Assistant
# Navigation + chatbot + recommender system merged into one Colab-ready script.


# ============================================================
# CONFIG
# ============================================================
CELL_SIZE = 0.2
MEMORY_FILE = "/content/navigation_memory.json"
GRID_0_FILE = "/content/floor_3_grid.npy"
GRID_1_FILE = "/content/floor_4_grid.npy"

# ============================================================
# DATA LOADING
# ============================================================
def load_json_file(path, default):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default

categories_data = load_json_file(CATEGORIES_FILE, [])
products_data = load_json_file(PRODUCTS_FILE, [])

# ============================================================
# STORE / ROOM MAPPING
# ============================================================
ROOM_INFO = {
    350: "Smart Devices Hub",
    351: "Computer Systems Hub",
    352: "Mobile & Tablets Hub",
    353: "Storage",
    354: "Smart Home Automation",
    355: "Smart Wearables",
    356: "Gaming",
    357: "Laptop Accessories",
    358: "Health & Personal Care",
    450: "Bedroom",
    451: "Living Room",
    452: "Study/Office",
    453: "Kitchen",
    454: "Dining Room",
    455: "Bathroom",
    456: "Furnishings",
    457: "Kitchen and Dining",
    458:"Home Decor",
    459: "Tools and utility",
    460: "Lighting and Electricals",
    461: "Cleaning and Bath",
    462:"Pet and Gardening",
}

STORE_CLUSTER_ROOM = {
    "Smart Devices Hub": 350,
    "Computer Systems Hub": 351,
    "Mobile & Tablets Hub": 352,
    "Storage":353,
    "Smart Home Automation":354,
    "Smart Wearables": 355,
    "Gaming": 356,
    "Laptop Accessories":357,
    "Health & Personal Care": 358,
    "Bedroom": 450,
    "Living Room": 451,
    "Study/Office": 452,
    "Kitchen": 453,
    "Dining Room": 454,
    "Bathroom": 455,
    "Furnishings": 456,
    "Kitchen and Dining": 457,
    "Home Decor": 458,
    "Tools and utility": 459,
    "Lighting and Electricals": 460,
    "Cleaning and Bath": 461,
    "Pet and Gardening": 462,
}

STORE_CLUSTER = {
    "Audio": "Smart Devices Hub",
    "Cameras": "Smart Devices Hub",
    "Computer Peripherals": "Computer Systems Hub",
    "Laptop & Desktop": "Computer Systems Hub",
    "Mobiles": "Mobile & Tablets Hub",
    "Tabelts": "Mobile & Tablets Hub",
}

SUB_TO_CATEGORY = {}
for type_block in categories_data:
    for cat in type_block.get("categories", []):
        for sub in cat.get("subCategories", []):
            SUB_TO_CATEGORY[sub] = cat["category"]

NAME_TO_ROOM = {name.lower(): room for room, name in ROOM_INFO.items()}

# ============================================================
# MULTILINGUAL PRODUCT & STORE MAPPING
# ============================================================
PRODUCT_TO_ROOM = {

    # =========================================================
    # 350 - Smart Devices Hub
    # =========================================================
    "camera": 350, "cameras": 350,
    "headphones": 350, "earbuds": 350,
    "speaker": 350, "smart device": 350,
    "iot": 350, "smart sensor": 350,
    "كاميرا": 350, "كاميرات": 350,
    "سماعات": 350, "سماعة": 350,
    "جهاز ذكي": 350, "انترنت الاشياء": 350,

    # =========================================================
    # 351 - Computer Systems Hub
    # =========================================================
    "computer": 351, "desktop": 351,
    "pc": 351, "server": 351,
    "motherboard": 351, "cpu": 351,
    "processor": 351, "gpu": 351,
    "ram": 351, "computer system": 351,
    "كمبيوتر": 351, "حاسب": 351,
    "معالج": 351, "رام": 351,
    "كارت شاشة": 351,

    # =========================================================
    # 352 - Mobile & Tablets Hub
    # =========================================================
    "mobile": 352, "phone": 352,
    "smartphone": 352, "tablet": 352,
    "ipad": 352, "iphone": 352,
    "samsung": 352, "charger": 352,
    "power bank": 352,
    "موبايل": 352, "هاتف": 352,
    "تليفون": 352, "تابلت": 352,
    "ايفون": 352, "سامسونج": 352,
    "شاحن": 352,

    # =========================================================
    # 353 - Storage
    # =========================================================
    "hard drive": 353, "ssd": 353,
    "hdd": 353, "usb": 353,
    "flash": 353, "memory": 353,
    "storage": 353, "sd card": 353,
    "external drive": 353,
    "هارد": 353, "فلاشه": 353,
    "ذاكرة": 353, "تخزين": 353,
    "كارت ميموري": 353,
    "RAM":353,"ram":353,"رامات":353,"راما":353,"رام":353,
    "Hard disk":353,"HDD":353,"HDD drive":353,"SSD":353,
    "SSD drive":353,"Mobile Memory Card":353,"Pen drive":353,
    "ليدر":353,"ريدر":353,"reader":353,"Reader":353,

    # =========================================================
    # 354 - Smart Home Automation
    # =========================================================
    "smart home": 354, "automation": 354,
    "smart lock": 354, "smart light": 354,
    "security camera": 354,
    "smart switch": 354,
    "بيت ذكي": 354,
    "قفل ذكي": 354,
    "كاميرا مراقبة": 354,
    "اضاءة ذكية": 354,

    # =========================================================
    # 355 - Smart Wearables
    # =========================================================
    "smart watch": 355, "smartwatch": 355,
    "wearable": 355, "fitness tracker": 355,
    "vr": 355, "ar glasses": 355,
    "ساعة ذكية": 355,
    "ويرابل": 355,
    "نظارة ذكية": 355,

    # =========================================================
    # 356 - Gaming
    # =========================================================
    "gaming": 356, "game": 356,
    "playstation": 356, "xbox": 356,
    "console": 356, "controller": 356,
    "gaming chair": 356,
    "gaming mouse": 356,
    "جيمينج": 356,
    "بلايستيشن": 356,
    "اكس بوكس": 356,
    "كونسول": 356,
    "كرسي جيمينج": 356,

    # =========================================================
    # 357 - Laptop Accessories
    # =========================================================
    "laptop": 357, "mouse": 357,
    "keyboard": 357, "laptop stand": 357,
    "dock": 357, "webcam": 357,
    "cooling pad": 357,
    "لابتوب": 357,
    "كيبورد": 357,
    "ماوس": 357,
    "ويب كام": 357,

    # =========================================================
    # 358 - Health & Personal Care
    # =========================================================
    "health": 358, "personal care": 358,
    "skincare": 358, "hair dryer": 358,
    "massager": 358,
    "electric toothbrush": 358,
    "صحة": 358,
    "عناية شخصية": 358,
    "مجفف شعر": 358,
    "فرشاة اسنان": 358,

    # =========================================================
    # 450 - Bedroom
    # =========================================================
    "bed": 450, "mattress": 450,
    "pillow": 450, "wardrobe": 450,
    "bedroom": 450,
    "سرير": 450,
    "مرتبة": 450,
    "مخدة": 450,
    "دولاب": 450,

    # =========================================================
    # 451 - Living Room
    # =========================================================
    "sofa": 451, "couch": 451,
    "tv": 451, "television": 451,
    "coffee table": 451,
    "living room": 451,
    "كنبة": 451,
    "صالون": 451,
    "تلفزيون": 451,
    "ركنة": 451,

    # =========================================================
    # 452 - Study/Office
    # =========================================================
    "desk": 452, "office chair": 452,
    "bookshelf": 452,
    "study": 452, "office": 452,
    "printer": 452,
    "مكتب": 452,
    "كرسي مكتب": 452,
    "مكتبة": 452,
    "طابعة": 452,

    # =========================================================
    # 453 - Kitchen
    # =========================================================
    "kitchen": 453, "fridge": 453,
    "refrigerator": 453,
    "oven": 453, "microwave": 453,
    "air fryer": 453,
    "blender": 453,
    "مطبخ": 453,
    "ثلاجة": 453,
    "بوتاجاز": 453,
    "ميكروويف": 453,

    # =========================================================
    # 454 - Dining Room
    # =========================================================
    "dining": 454,
    "dining table": 454,
    "dining chair": 454,
    "buffet": 454,
    "سفرة": 454,
    "ترابيزة سفرة": 454,
    "غرفة سفرة": 454,

    # =========================================================
    # 455 - Bathroom
    # =========================================================
    "bathroom": 455,
    "toilet": 455,
    "shower": 455,
    "sink": 455,
    "towel": 455,
    "حمام": 455,
    "دوش": 455,
    "مغسلة": 455,

    # =========================================================
    # 456 - Furnishings
    # =========================================================
    "curtain": 456,
    "carpet": 456,
    "rug": 456,
    "bedsheet": 456,
    "blanket": 456,
    "ستاير": 456,
    "سجاد": 456,
    "مفروشات": 456,

    # =========================================================
    # 457 - Kitchen and Dining
    # =========================================================
    "plate": 457,
    "cup": 457,
    "cutlery": 457,
    "cookware": 457,
    "pot": 457,
    "طبق": 457,
    "كوب": 457,
    "حلل": 457,

    # =========================================================
    # 458 - Home Decor
    # =========================================================
    "decor": 458,
    "vase": 458,
    "mirror": 458,
    "painting": 458,
    "clock": 458,
    "ديكور": 458,
    "تحف": 458,
    "مراية": 458,

    # =========================================================
    # 459 - Tools and Utility
    # =========================================================
    "tool": 459,
    "drill": 459,
    "hammer": 459,
    "screwdriver": 459,
    "toolbox": 459,
    "عدة": 459,
    "شنيور": 459,
    "مفك": 459,

    # =========================================================
    # 460 - Lighting and Electricals
    # =========================================================
    "light": 460,
    "lamp": 460,
    "chandelier": 460,
    "bulb": 460,
    "led": 460,
    "لمبة": 460,
    "نجف": 460,
    "اضاءة": 460,

    # =========================================================
    # 461 - Cleaning and Bath
    # =========================================================
    "cleaning": 461,
    "vacuum": 461,
    "detergent": 461,
    "mop": 461,
    "broom": 461,
    "تنظيف": 461,
    "مكنسة": 461,
    "ممسحة": 461,

    # =========================================================
    # 462 - Pet and Gardening
    # =========================================================
    "pet": 462,
    "dog": 462,
    "cat": 462,
    "plant": 462,
    "garden": 462,
    "flower": 462,
    "حيوان": 462,
    "كلب": 462,
    "قطة": 462,
    "زرع": 462,
}

STORE_ALIASES = {
    350: ["Smart Devices Hub", "smart devices", "smart", "devices", "audio", "cameras", "wearables", "سمارت", "أجهزة ذكية", "كاميرات", "سماعات", "ساعات ذكية"],

    351: ["Computer Systems Hub", "computer", "computers", "laptop", "desktop", "pc", "كمبيوتر", "لابتوب", "لاب توب", "كمبيوترات"],

    352: ["Mobile & Tablets Hub", "mobile", "tablet","Tabelts", "phone", "phones", "smartphone", "موبايل", "موبايلات", "تابلت", "تليفون", "هواتف"],

    353: ["Storage", "storage", "ssd", "hdd", "flash", "memory", "تخزين", "هارد", "فلاش"],

    354: ["Smart Home Automation", "smart home", "automation", "iot", "home control", "منزل ذكي", "أتمتة"],

    355: ["Smart Wearables", "wearables", "smartwatch", "watch", "fitness band", "ساعات ذكية", "أجهزة قابلة للارتداء"],

    356: ["Gaming", "gaming", "games", "playstation", "xbox", "nintendo", "ألعاب", "جيمينج", "بلايستيشن"],

    357: ["Laptop Accessories", "laptop accessories", "charger", "mouse", "keyboard", "laptop bag", "إكسسوارات لابتوب", "شاحن", "ماوس"],

    358: ["Health & Personal Care", "health", "personal care", "beauty", "skincare", "صحة", "عناية شخصية", "جمال", "بشرة"],

    450: ["Bedroom", "bedroom", "bed", "sleep", "غرفة نوم", "نوم", "سرير"],

    451: ["Living Room", "living room", "living", "sofa", "couch", "صالة", "ركنة", "انتريه", "صالون"],

    452: ["Study/Office", "study", "office", "desk", "work", "مكتب", "دراسة", "مذاكرة"],

    453: ["Kitchen", "kitchen", "cook", "cooking", "مطبخ", "طبخ", "أكل"],

    454: ["Dining Room", "dining", "dining room", "table", "سفرة", "غرفة سفرة"],

    455: ["Bathroom", "bathroom", "bath", "shower", "حمام", "دوش"],

    456: ["Furnishings", "furnishings", "curtains", "carpet", "rug", "مفروشات", "ستاير", "سجاد"],

    457: ["Kitchen and Dining", "cookware", "utensils", "pots", "pans", "أدوات مطبخ", "حلل", "معالق"],

    458: ["Home Decor", "home decor", "decor", "decoration", "ديكور", "تحف"],

    459: ["Tools and Utility", "tools", "utility", "hardware", "عدة", "أدوات"],

    460: ["Lighting and Electricals", "lighting", "lights", "electrical", "lamps", "اضاءة", "لمبات", "نجف"],

    461: ["Cleaning and Bath", "cleaning", "bath products", "detergent", "تنظيف", "منظفات"],

    462: ["Pet and Gardening", "pet", "pets", "gardening", "garden", "plants", "حيوانات", "حديقة", "زرع"],
}

def build_alias_to_room():
    mapping = {}
    for room, aliases in STORE_ALIASES.items():
        for alias in aliases:
            mapping[alias.lower()] = room
    return mapping

ALIAS_TO_ROOM = build_alias_to_room()

# ============================================================
# GRID / CENTROIDS
# ============================================================
grid_0 = np.load(GRID_0_FILE)
grid_1 = np.load(GRID_1_FILE)

room_centroids_f3_grid = {
    350: (70, 465),
    351: (70, 435),
    352: (70, 395),
    353: (70, 360),
    354: (70, 320),
    355: (70, 285),
    356: (70, 215),
    357: (70, 160),
    358: (70, 85),
}

# FIX 1: Floor 4 centroids added explicitly
room_centroids_f4_grid = {
    450: (80, 460),
    451: (80, 430),
    452: (80, 392),
    453: (80, 350),
    454: (80, 310),
    455: (80, 272),
    456: (80, 235),
    457: (80, 200),
    458: (80, 165),
    459: (80, 130),
    460: (80, 95),
    461: (80, 60),
    462: (80, 25),

}


# FIX 2: Each grid uses its own shape (grids may differ in size)
def apply_room_clearance():
    # Floor 3 clearance on grid_0
    for r, c in room_centroids_f3_grid.values():
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                rr, cc = r + dr, c + dc
                if 0 <= rr < grid_0.shape[0] and 0 <= cc < grid_0.shape[1]:
                    grid_0[rr, cc] = 0
    # Floor 4 clearance on grid_1
    for r, c in room_centroids_f4_grid.values():
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                rr, cc = r + dr, c + dc
                if 0 <= rr < grid_1.shape[0] and 0 <= cc < grid_1.shape[1]:
                    grid_1[rr, cc] = 0


apply_room_clearance()


def reset_grids():
    global grid_0, grid_1
    grid_0 = np.load(GRID_0_FILE)
    grid_1 = np.load(GRID_1_FILE)
    apply_room_clearance()

# ============================================================
# MEMORY LOG
# ============================================================
def load_memory():
    try:
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return []


def save_memory(memory):
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(memory, f, indent=4)


user_memory = load_memory()
navigation_counter = len(user_memory) + 1


def save_navigation_mem(start_room, dest_room):
    global navigation_counter
    entry = {
        "nav_id": f"N{navigation_counter}",
        "start_room": start_room,
        "dest_room": dest_room,
    }
    user_memory.append(entry)
    save_memory(user_memory)
    navigation_counter += 1
    return entry

# ============================================================
# SYSTEM PROMPT (for OpenRouter fallback only)
# ============================================================
SYSTEM_PROMPT = """
You are an AI assistant for indoor navigation inside a shopping mall.

Mall stores:
350 Smart Devices Hub
351 Computer Systems Hub
352 Mobile & Tablets Hub
353 Storage
354 Smart Home Automation
355 Smart Wearables
356 Gaming
357 Laptop Accessories
358 Health & Personal Care
450 Bedroom
451 Living Room
452 Study/Office
453 Kitchen
454 Dining Room
455 Bathroom
456 Furnishings
457 Kitchen and Dining
458 Home Decor
459 Tools and utility
460 Lighting and Electricals
461 Cleaning and Bath
462 Pet and Gardening

Rules:
- If the user asks about a store, answer with store information.
- If the user asks for navigation, help them navigate.
- If the user says he is hungry, suggest kitchen / dining / food-related stores and when he tells you that he agree os suggestion then help him navigate.
- Keep answers concise and helpful.
"""

# ============================================================
# HELPERS / SESSION STATE
# ============================================================
def set_session_defaults(session):
    session = session or {}
    defaults = {
        "username": None,
        "gender": "other",
        "low": 500,
        "high": 5000,
        "is_new": True,
        "start_floor": None,
        "start_xy": None,
        "nav_mode": "Normal",
        "recommended_room": None,
        "nav_target_room": None,
        "last_dest_room": None,
        "last_referenced_room": None,
        "selected_subcategory": None,
        "accepted_store": False,
        "special_needs": False,
        "chat_start_floor": None,
        "chat_start_xy": None,
        "chat_dest_room": None,
    }
    for k, v in defaults.items():
        session.setdefault(k, v)
    return session


def get_floor(room):
    if 350 <= room <= 358:
        return 0
    if 450 <= room <= 462:
        return 1
    return 0


def get_centroid(room):
    if room in room_centroids_f3_grid:
        return room_centroids_f3_grid[room]
    if room in room_centroids_f4_grid:
        return room_centroids_f4_grid[room]
    return None


def clamp_point(x, y, grid):
    r = int(max(0, min(grid.shape[0] - 1, round(float(x)))))
    c = int(max(0, min(grid.shape[1] - 1, round(float(y)))))
    return r, c


def get_store_info(room):
    info = {
        350: "Smart devices, smart home gadgets, IoT products, and electronics.",
        351: "Computer systems including desktops, laptops, and PC components.",
        352: "Mobile & tablets including smartphones and accessories.",
        353: "Storage devices like SSDs, HDDs, flash drives, and memory cards.",
        354: "Smart home automation devices and IoT control systems.",
        355: "Smart wearables including smartwatches and fitness bands.",
        356: "Gaming consoles, games, and gaming accessories.",
        357: "Laptop accessories like chargers, bags, mice, and keyboards.",
        358: "Health and personal care products including beauty and wellness items.",

        450: "Bedroom furniture and sleep essentials.",
        451: "Living room furniture and decor items.",
        452: "Study and office furniture and productivity tools.",
        453: "Kitchen appliances and cooking essentials.",
        454: "Dining room furniture and dining essentials.",
        455: "Bathroom fixtures and bathroom essentials.",
        456: "Home furnishings like curtains, carpets, and textiles.",
        457: "Kitchen and dining tools, utensils, and cookware.",
        458: "Home decor items and decorative accessories.",
        459: "Tools and utility equipment for home use.",
        460: "Lighting systems and electrical products.",
        461: "Cleaning supplies and bath-related products.",
        462: "Pet supplies and gardening products."
    }

    name = ROOM_INFO.get(room)
    if not name:
        return None

    return f"🏪 {name} (Room {room})\n📝 {info.get(room, 'Mall store information is available for this store.')}"

# ============================================================
# ENHANCED ROOM FINDING (Multilingual)
# ============================================================
def find_room_in_text(text):
    text_l = (text or "").lower()
    # Direct room numbers
    room_match = re.search(r"\b(3\d{2}|4[5-6]\d)\b", text_l)
    if room_match:
        room = int(room_match.group(1))
        if room in ROOM_INFO:
            return room
    # Store names (longest match first)
    for name, room in sorted(NAME_TO_ROOM.items(), key=lambda x: len(x[0]), reverse=True):
        if name in text_l:
            return room
    # Aliases
    for alias, room in sorted(ALIAS_TO_ROOM.items(), key=lambda x: len(x[0]), reverse=True):
        if alias in text_l:
            return room
    # Product keywords
    for keyword, room in sorted(PRODUCT_TO_ROOM.items(), key=lambda x: len(x[0]), reverse=True):
        if keyword in text_l:
            return room
    return None


def find_rooms_in_text(text):
    text_l = (text or "").lower()
    found = re.findall(r"\b(3\d{2}|4[5-6]\d)\b", text_l)
    return [int(r) for r in found if int(r) in ROOM_INFO]


def extract_start_and_dest(text):
    """Extract start and destination from text like 'from 350 to 352' or 'from room 350 to room 352'"""
    text_l = str(text or "").lower()
    rooms = find_rooms_in_text(text)

    start_room = None
    dest_room = None

    # Pattern: from X to Y / من X إلى Y
    from_to_patterns = [
        r"(?:from|من)\s+(?:room\s+)?(\d{3}).*?(?:to|إلى|الي|لـ|ل|toward|towards)\s+(?:room\s+)?(\d{3})",
        r"(?:from|من)\s+(?:room\s+)?(\d{3}).*?(\d{3})",
    ]

    for pattern in from_to_patterns:
        match = re.search(pattern, text_l)
        if match:
            start_room = int(match.group(1))
            dest_room = int(match.group(2))
            break

    # If no regex match, use heuristic
    if start_room is None and len(rooms) >= 2:
        if "from" in text_l or "من" in text:
            start_room = rooms[0]
            dest_room = rooms[1]
        elif any(w in text_l for w in ["to", "إلى", "الي"]):
            start_room = rooms[0]
            dest_room = rooms[1]
        else:
            dest_room = rooms[-1]
    elif start_room is None and len(rooms) == 1:
        dest_room = rooms[0]

    # If still no destination, try product/alias matching
    if dest_room is None:
        dest_room = find_room_in_text(text)

    return start_room, dest_room


def get_intro_message():
    return (
        "Hello 👋 Welcome to the Mall Navigation Assistant.\n"
        "You can navigate to:\n"
        + "\n".join([f"- {name} ({room})" for room, name in ROOM_INFO.items()])
        + "\nIf you're hungry, you can go to the kitchen/dining or cafeteria-related stores."
    )


def local_chat_reply(user_text):
    low = (user_text or "").lower().strip()
    if not low:
        return "Please type a message or ask about a store."
    if any(word in low for word in ["hi", "hello", "hey", "good morning", "good evening", "how are you"]):
        return "Hello 👋 Welcome to SmartMall. Tell me a store name, a room number, or where you want to go."
    if any(word in low for word in ["thanks", "thank you"]):
        return "You are welcome."

    room = find_room_in_text(user_text)
    if room is not None and any(k in low for k in ["what", "tell", "about", "info", "describe", "where"]):
        info = get_store_info(room)
        if info:
            return info

    return None


# ============================================================
# AUTH / DATABASE OPERATIONS (FIXED with special_needs)
# ============================================================
def save_preference(username, preference):
    if not preference or len(preference) != 3:
        return
    main, cat, sub = preference
    cursor.execute(
        """INSERT INTO user_preferences (username, main_category, category, subcategory)
           VALUES (?, ?, ?, ?)""",
        (username, main, cat, sub),
    )
    conn.commit()


def entry_point():
    print("\n=== Smart Mall System ===")
    while True:
        print("\n1) Signup")
        print("2) Login")
        choice = input("Enter choice: ").strip()
        if choice == "1":
            return "signup"
        if choice == "2":
            return "login"
        print("Invalid input ❌")


def signup():
    print("\n=== SIGNUP ===")
    while True:
        username = input("Choose username: ").strip()
        cursor.execute("SELECT username FROM users WHERE username=?", (username,))
        if cursor.fetchone():
            print("❌ Username exists, try again\n")
        else:
            break

    password = input("Choose password: ").strip()
    name = input("Enter name: ").strip()
    age = int(input("Enter age: "))
    gender = input("Enter gender (male/female): ").strip().lower()

    auto = input("Use auto budget? (yes/no): ").strip().lower()
    if auto == "no":
        lower = float(input("Lower budget: "))
        upper = float(input("Upper budget: "))
    else:
        lower, upper = 500, 5000

    # FIXED: Save special_needs to database
    special = input("Are you a person with special needs? (yes/no): ").strip().lower()
    special_flag = 1 if special == "yes" else 0

    cursor.execute(
        "INSERT INTO users VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
        (username, password, name, age, gender, lower, upper, special_flag),
    )
    conn.commit()

    print("\n✅ Signup successful")
    return username, gender, lower, upper, None, special_flag


def login():
    print("\n=== LOGIN ===")
    while True:
        username = input("Username: ").strip()
        password = input("Password: ").strip()

        cursor.execute(
            "SELECT * FROM users WHERE username=? AND password=?",
            (username, password),
        )
        user = cursor.fetchone()

        if user:
            print("✅ Login successful")
            break
        print("❌ Invalid credentials, try again\n")

    cursor.execute(
        """SELECT main_category, category, subcategory
           FROM user_preferences
           WHERE username=?
           ORDER BY id DESC LIMIT 1""",
        (username,),
    )
    pref = cursor.fetchone()
    preference = pref if pref else None

    # FIXED: Retrieve special_needs from database (index 7)
    special_flag = bool(user[7]) if len(user) > 7 else False

    return username, user[4], user[5], user[6], preference, special_flag


def auth_system():
    while True:
        mode = entry_point()
        if mode == "signup":
            user_data = signup()
            if user_data:
                return user_data, True
        elif mode == "login":
            user_data = login()
            if user_data:
                return user_data, False
        print("❌ Authentication failed, try again...\n")


def get_ble_location():
    print("\n📍 Enter your current location")
    x = float(input("X coordinate: "))
    y = float(input("Y coordinate: "))
    return (x, y)


def save_navigation(username, start_room, dest_room):
    if isinstance(start_room, tuple):
        start_room = f"{start_room[0]},{start_room[1]}"
    cursor.execute(
        "INSERT INTO history (username, start_point, end_store) VALUES (?, ?, ?)",
        (username, start_room, str(dest_room)),
    )
    conn.commit()

# ============================================================
# STORE RECOMMENDATION
# ============================================================
def nearest_store(user_pos, user_floor):
    best_room = None
    best_dist = float("inf")

    for _, room in STORE_CLUSTER_ROOM.items():
        # ✅ فلترة على نفس الدور
        if get_floor(room) != user_floor:
            continue

        centroid = get_centroid(room)
        if centroid is None:
            continue

        dist = np.sqrt((user_pos[0] - centroid[0]) ** 2 + (user_pos[1] - centroid[1]) ** 2)

        if dist < best_dist:
            best_dist = dist
            best_room = room

    return best_room


def most_visited_store(username):
    cursor.execute(
        """SELECT end_store, COUNT(*) as cnt
           FROM history
           WHERE username=?
           GROUP BY end_store
           ORDER BY cnt DESC""",
        (username,),
    )
    result = cursor.fetchone()
    return int(result[0]) if result else None


# ============================================================
# PRODUCT RECOMMENDER
# ============================================================
def clean_price(price_str):
    digits = "".join(filter(str.isdigit, str(price_str)))
    return int(digits) if digits else 0


def get_categories_for_type(type_name):
    for t in categories_data:
        if t.get("type") == type_name:
            return [c.get("category") for c in t.get("categories", [])]
    return []


def get_subcategories(type_name, category_name):
    for t in categories_data:
        if t.get("type") == type_name:
            for c in t.get("categories", []):
                if c.get("category") == category_name:
                    return c.get("subCategories", [])
    return []


def resolve_store_from_category(parent_category):
    store_name = STORE_CLUSTER.get(parent_category, parent_category)
    room = STORE_CLUSTER_ROOM.get(store_name)
    return store_name, room


def category_flow():
    print("\n🧭 Category Selection")

    print("\nMain categories:")
    for i, c in enumerate(categories_data):
        print(i, c["type"])
    main = int(input("Select main category: "))
    main_cat = categories_data[main]

    print("\nSub categories:")
    for i, c in enumerate(main_cat["categories"]):
        print(i, c["category"])
    sub = int(input("Select category: "))
    sub_cat = main_cat["categories"][sub]

    print("\nSub-sub categories:")
    for i, c in enumerate(sub_cat["subCategories"]):
        print(i, c)
    sub_sub = int(input("Select subcategory: "))

    selected_subcategory = sub_cat["subCategories"][sub_sub]
    if selected_subcategory is None:
        print("❌ No subcategory selected")
        return None, None, None, None

    main_category = main_cat["type"]
    parent_category = sub_cat["category"]
    _, end_store = resolve_store_from_category(parent_category)
    return selected_subcategory, end_store, parent_category, main_category


def get_budget(default_low, default_high):
    print("\n💰 Budget Selection")
    choice = input("Use auto budget? (yes/no): ").strip().lower()
    if choice == "no":
        lower = float(input("Enter lower budget: "))
        upper = float(input("Enter upper budget: "))
    else:
        lower, upper = default_low, default_high
    return lower, upper


def get_products(subcategory):
    if not subcategory:
        return []

    result = []
    subcategory = str(subcategory).strip().lower()

    for type_block in products_data:
        for cat in type_block.get("items", []):
            cat_name = cat.get("category")
            for sub in cat.get("items", []):
                sub_name = sub.get("subCategory")
                if cat_name and cat_name.lower() == subcategory:
                    result.extend(sub.get("items", []))
                elif sub_name and sub_name.lower() == subcategory:
                    result.extend(sub.get("items", []))
    return result


def filter_by_budget(products, low, high):
    filtered = []
    for p in products:
        price = clean_price(p.get("Price", 0))
        if low <= price <= high:
            item = dict(p)
            item["price_int"] = price
            filtered.append(item)
    return filtered


def gender_filter(products, gender):
    if gender not in ["male", "female"]:
        return products

    keywords = {
        "male": ["gaming", "tool", "power", "sports", "fitness"],
        "female": ["beauty", "hair", "skin", "cosmetic", "makeup", "fashion"],
    }

    preferred, others = [], []
    for p in products:
        name = str(p.get("Name", "")).lower()
        if any(k in name for k in keywords[gender]):
            preferred.append(p)
        else:
            others.append(p)
    return preferred + others


def add_promotions(products):
    result = []
    for p in products:
        discount = random.randint(10, 50)
        original = clean_price(p.get("Price", 0))
        discounted = int(original * (1 - discount / 100))
        result.append({
            "Name": p.get("Name", "Unknown Product"),
            "Brand": p.get("Brand", "N/A"),
            "Original": f"{original}",
            "Discounted": f"{discounted}",
            "Discount": discount,
            "Rating": p.get("Rating", 0),
            "Reviews": p.get("No of Reviews", 0),
        })
    return result


def recommend_for_user(selected_subcategory, gender, low_budget, high_budget):
    products = get_products(selected_subcategory)
    if not products:
        return {"store": "Unknown", "room": None, "products": []}

    products = filter_by_budget(products, low_budget, high_budget)
    if not products:
        products = get_products(selected_subcategory)

    products = gender_filter(products, gender)
    products = sorted(products, key=lambda x: x.get("Rating", 0), reverse=True)
    top_products = add_promotions(products[:5])

    parent_category = SUB_TO_CATEGORY.get(selected_subcategory)
    store_name = STORE_CLUSTER.get(parent_category, parent_category) if parent_category else None
    room = STORE_CLUSTER_ROOM.get(store_name) if store_name else None

    return {"store": store_name or "Unknown", "room": room, "products": top_products}

# ============================================================
# NAVIGATION / A*
# ============================================================
def find_nearest_free(grid, r, c):
    free = (grid == 0)
    _, ind = distance_transform_edt(~free, return_indices=True)
    nr, nc = ind[:, r, c]
    return int(nr), int(nc)


def extract_stairs(grid):
    binary = (grid == 2).astype(int)
    labeled, num = label(binary)
    return labeled, num


stairs_0, n0 = extract_stairs(grid_0)
stairs_1, n1 = extract_stairs(grid_1)
stairs = {}
stair_cells = {0: {}, 1: {}}
for i in range(1, min(n0, n1) + 1):
    sid = f"S{i}"
    stairs[sid] = {"floors": [0, 1]}
    for r, c in np.argwhere(stairs_0 == i):
        stair_cells[0][(int(r), int(c))] = sid
    for r, c in np.argwhere(stairs_1 == i):
        stair_cells[1][(int(r), int(c))] = sid

elevator_cells = {0: set(), 1: set()}
for f in [0, 1]:
    grid = grid_0 if f == 0 else grid_1
    coords = np.argwhere(grid == 3)
    for r, c in coords:
        elevator_cells[f].add((int(r), int(c)))


# FIX 3: Helper to find nearest elevator on a given floor, then land on corridor
def find_nearest_elevator(floor, r, c):
    cells = list(elevator_cells.get(floor, []))
    if not cells:
        return None
    return min(cells, key=lambda p: (p[0] - r) ** 2 + (p[1] - c) ** 2)


fire_active = False
fire_room = None
fire_step = 0
fire_center = None
crowded_active = False
crowded_room = None
crowded_center = None


def mark_danger(grid, centroid, radius=20):
    r0, c0 = centroid
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            dist = np.sqrt((i - r0) ** 2 + (j - c0) ** 2)
            if dist < radius:
                grid[i, j] = 1


def mark_crowd(grid, centroid, room_radius=15, corridor_radius=40):
    r0, c0 = centroid
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            dist = np.sqrt((i - r0) ** 2 + (j - c0) ** 2)
            if dist < room_radius:
                grid[i, j] = 5
            elif dist < corridor_radius and grid[i, j] == 0:
                grid[i, j] = 6


def heuristic(a, b):
    return abs(a[1] - b[1]) + abs(a[2] - b[2]) + abs(a[0] - b[0]) * 10


def neighbors(node, mode):
    floor, r, c = node
    grid = grid_0 if floor == 0 else grid_1
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    result = []

    for dr, dc in moves:
        nr = r + dr
        nc = c + dc
        if 0 <= nr < grid.shape[0] and 0 <= nc < grid.shape[1]:
            cell = grid[nr, nc]
            if cell == 1:
                continue

            cost = 1

            if fire_active and fire_center is not None:
                fr, fc = fire_center
                dist = np.sqrt((nr - fr) ** 2 + (nc - fc) ** 2)
                if dist < 120:
                    cost += (120 - dist) * 0.3

            if cell == 4:
                cost += 8

            if mode == "Crowded":
                if cell == 5:
                    cost += 200
                elif cell == 6:
                    cost += 40

            if mode == "Special Needs" and cell == 4:
                cost += 100

            result.append(((floor, int(nr), int(nc)), cost))

    is_stair = (r, c) in stair_cells[floor]
    is_elevator = (r, c) in elevator_cells[floor]

    if mode == "Special Needs":
        if is_elevator:
            for f in [0, 1]:
                if f != floor:
                    # FIX 3: land on corridor near the destination-floor elevator
                    nearest_elev = find_nearest_elevator(f, r, c)
                    if nearest_elev:
                        er, ec = nearest_elev
                        dest_grid = grid_0 if f == 0 else grid_1
                        cr, cc = find_nearest_free(dest_grid, er, ec)
                        result.append(((f, cr, cc), 10))
    else:
        if is_stair:
            sid = stair_cells[floor][(r, c)]
            cost = 15 if mode != "Crowded" else 40
            if fire_active and fire_center is not None:
                fr, fc = fire_center
                dist = np.sqrt((r - fr) ** 2 + (c - fc) ** 2)
                if dist < 150:
                    cost += 200
            for f in stairs[sid]["floors"]:
                if f != floor:
                    result.append(((f, r, c), cost))

    return result


def astar(start, goal, mode="Normal"):
    open_set = []
    heapq.heappush(open_set, (0, start))
    came = {}
    g = {start: 0}

    while open_set:
        _, cur = heapq.heappop(open_set)
        if cur == goal:
            path = []
            while cur in came:
                path.append(cur)
                cur = came[cur]
            path.append(start)
            return path[::-1]

        for nxt, cost in neighbors(cur, mode):
            ng = g[cur] + cost
            if nxt not in g or ng < g[nxt]:
                g[nxt] = ng
                heapq.heappush(open_set, (ng + heuristic(nxt, goal), nxt))
                came[nxt] = cur
    return None


def path_to_instructions(path):
    if not path:
        return "No path found."

    instructions = []
    last_floor, last_r, last_c = path[0]
    current_dir = None
    dist = 0

    def add_instruction(direction, distance):
        if distance > 0:
            instructions.append(f"Move {direction} {distance * CELL_SIZE:.1f}m.")

    for i in range(1, len(path)):
        f, r, c = path[i]
        if f != last_floor:
            add_instruction(current_dir, dist)
            instructions.append(f"Take stairs from floor {last_floor + 3} to floor {f + 3}.")
            current_dir = None
            dist = 0
            last_floor = f

        dr = r - last_r
        dc = c - last_c
        if abs(dr) > abs(dc):
            direction = "down" if dr > 0 else "up"
            step = abs(dr)
        else:
            direction = "right" if dc > 0 else "left"
            step = abs(dc)

        if direction == current_dir:
            dist += step
        else:
            add_instruction(current_dir, dist)
            current_dir = direction
            dist = step

        last_r, last_c = r, c

    add_instruction(current_dir, dist)
    instructions.append("You have arrived at your destination room.")
    return ". ".join(instructions)


def plot_static_path(path):
    fig, axs = plt.subplots(1, 2, figsize=(18, 6))
    grids = [grid_0, grid_1]

    for i, grid in enumerate(grids):
        axs[i].imshow(grid, cmap="gray_r", origin="upper")
        stairs_pos = np.where(grid == 2)
        axs[i].scatter(stairs_pos[1], stairs_pos[0], s=40, label="Stairs")

        xs = [p[2] for p in path if p[0] == i]
        ys = [p[1] for p in path if p[0] == i]
        if xs and ys:
            axs[i].plot(xs, ys, linewidth=3, label="Path")
            axs[i].scatter(xs[0], ys[0], s=80, label="Start")
            axs[i].scatter(xs[-1], ys[-1], s=80, label="End")

        axs[i].set_title(f"Floor {i + 3}")
        axs[i].legend()

    img_path = "/content/static_path.png"
    plt.savefig(img_path)
    plt.close()
    return img_path


def navigate(start_floor, start_xy, dest_room, mode="Normal"):
    if start_xy is None or dest_room is None:
        return None, "❌ Missing start or destination."

    sf = int(start_floor)
    sr, sc = clamp_point(start_xy[0], start_xy[1], grid_0 if sf == 0 else grid_1)
    sr, sc = find_nearest_free(grid_0 if sf == 0 else grid_1, sr, sc)

    df = get_floor(dest_room)
    dc = get_centroid(dest_room)
    if dc is None:
        return None, "❌ Could not find room centroid."

    gr, gc = clamp_point(dc[0], dc[1], grid_0 if df == 0 else grid_1)
    gr, gc = find_nearest_free(grid_0 if df == 0 else grid_1, gr, gc)

    path = astar((sf, sr, sc), (df, gr, gc), mode)
    instructions = path_to_instructions(path)
    img_path = plot_static_path(path) if path else None
    return img_path, instructions

# ============================================================
# FAST CHATBOT LOGIC (No LLM - Instant Response)
# ============================================================

def detect_language(text):
    if not text:
        return "en"
    if any("\u0600" <= c <= "\u06FF" for c in str(text)):
        return "ar"
    return "en"


def is_strong_navigation_request(text):
    """Check if user explicitly wants to navigate (strong intent)"""
    low = str(text or "").lower()
    strong_nav = [
        "go to", "take me to", "navigate to", "guide me to", "directions to", "path to",
        "how to reach", "send me to", "show me the way to", "walk me to", "move to",
        "i want to go", "want to go", "need to go", "get me to", "let's go", "lets go",
        "from", "من", "اذهب", "اخذني", "دلني", "طريق", "وديني", "عايز اروح", "عاوز اروح",
        "هيا بنا", "مشي", "روح", "إلى", "الي", "لـ", "لغرفة", "عايز انزل", "هنروح",
    ]
    return any(w in low for w in strong_nav)


def is_product_request(text):
    """Check if user is asking about a product"""
    low = str(text or "").lower()
    product_words = [
        "buy", "looking for", "where can i find", "i want", "searching", "do you have",
        "need", "want to purchase", "shopping", "get me", "find",
        "اشتري", "ابحث", "اين اجد", "عايز", "عاوز", "فين", "عندكم", "محتاج",
        "جيب", "هاشتري", "عايز انزل اشتري", "دور على", "لو سمحت"
    ]
    return any(w in low for w in product_words)


def is_agreement(text):
    """Check if user agrees to go somewhere"""
    low = str(text or "").lower().strip()
    # Strong agreement phrases
    phrases = [
        "take me there", "guide me there", "go there", "navigate me there",
        "lets go", "let's go", "yes take me", "yes navigate", "please do", "do it",
        "navigate", "mashi", "tamam",
        "اخذني هناك", "اذهب هناك", "هيا بنا", "تمام", "ماشي", "فضل",
        "يلا بينا", "وديني", "روح بيا", "هنروح", "كمل", "يسطا"
    ]
    if any(w in low for w in phrases):
        return True
    # Short words need to be standalone
    words = low.split()
    agree_words = ["ok", "okay", "yes", "sure", "go", "نعم", "اوكي"]
    return any(w in words for w in agree_words)


def is_info_request(text):
    """Check if user wants store info"""
    low = str(text or "").lower()
    info_words = [
        "what is", "tell me about", "info about", "describe", "where is", "about",
        "information", "ما هو", "اخبرني", "معلومات", "وصف", "اين", "عن", "مكان",
        "ايه", "ايه هو", "فين مكان", "عايز اعرف", "اقولي"
    ]
    return any(w in low for w in info_words)


def chatbot_respond(user_text, history, session):
    session = set_session_defaults(session)
    history = history or []
    if not history:
        history.append(("", get_intro_message()))

    lower = str(user_text or "").lower().strip()
    lang = detect_language(user_text)

    # --- 0. FAST GREETINGS ---
    if any(w in lower for w in ["hi", "hello", "hey", "good morning", "good evening"]):
        reply = "Hello 👋 Welcome to SmartMall! Tell me a store name, a room number, or where you want to go."
        history.append((user_text, reply))
        try: gTTS(reply).save("/content/voice.mp3")
        except: pass
        return history, None, "/content/voice.mp3", session

    if any(w in lower for w in ["مرحبا", "السلام عليكم", "هاي", "أهلا", "اهلا", "صباح الخير", "مساء الخير", "هلا"]):
        reply = "أهلاً بك 👋 أنا مساعد SmartMall. أخبرني باسم المتجر أو رقم الغرفة، أو إلى أين تريد الذهاب."
        history.append((user_text, reply))
        try: gTTS(reply, lang="ar").save("/content/voice.mp3")
        except: pass
        return history, None, "/content/voice.mp3", session

    # --- 1. EXTRACT START AND DESTINATION ---
    explicit_start, explicit_dest = extract_start_and_dest(user_text)

    # Also try to find any room mentioned
    any_room = find_room_in_text(user_text)
    if explicit_dest is None and any_room:
        explicit_dest = any_room

    # --- 2. CHECK AGREEMENT (for previous suggestions) ---
    is_agree = is_agreement(user_text)

    # If agreement and no explicit destination, use last referenced store
    if is_agree and explicit_dest is None:
        explicit_dest = (
            session.get("last_referenced_room")
            or session.get("last_dest_room")
            or session.get("nav_target_room")
            or session.get("chat_dest_room")
        )

    # --- 3. INTENT DETECTION ---
    is_strong_nav = is_strong_navigation_request(user_text)
    wants_info = is_info_request(user_text)
    wants_product = is_product_request(user_text)

    # If user explicitly asks to go somewhere (strong nav intent + dest) or agrees to previous suggestion
    wants_nav = is_strong_nav or (is_agree and explicit_dest is not None)

    # "go for" means shopping intent, not navigation
    if "go for" in lower:
        wants_nav = False
        wants_product = True

    # --- Handle explicit "no" ---
    if any(w in lower for w in ["no", "لا", "لأ", "مش دلوقتي", "not now"]):
        reply = "Okay, let me know if you need anything else." if lang != "ar" else "ماشي، قولي لو محتاج حاجة تانية."
        history.append((user_text, reply))
        try: gTTS(reply, lang="ar" if lang=="ar" else "en").save("/content/voice.mp3")
        except: pass
        return history, None, "/content/voice.mp3", session

    # --- 4. INFO ABOUT STORE (no navigation unless strong intent) ---
    if wants_info and explicit_dest and not is_strong_nav:
        info = get_store_info(explicit_dest)
        if info:
            store_name = ROOM_INFO.get(explicit_dest, f"Room {explicit_dest}")
            session["last_referenced_room"] = explicit_dest
            session["chat_dest_room"] = explicit_dest
            if lang == "ar":
                reply = f"{info}\n\nعايز أوديك هناك؟ قولي 'yes' أو 'ماشي'."
            else:
                reply = f"{info}\n\nWould you like me to take you there? Say 'yes' or 'ok'."
            history.append((user_text, reply))
            try: gTTS(reply, lang="ar" if lang=="ar" else "en").save("/content/voice.mp3")
            except: pass
            return history, None, "/content/voice.mp3", session

    # --- 5. PRODUCT QUERY (info only, no nav unless strong intent or agreement) ---
    if wants_product and explicit_dest and not is_strong_nav and not is_agree:
        store_name = ROOM_INFO.get(explicit_dest, f"Room {explicit_dest}")
        session["last_referenced_room"] = explicit_dest
        session["chat_dest_room"] = explicit_dest
        if lang == "ar":
            reply = f"ده موجود في {store_name} (غرفة {explicit_dest}). عايز أوديك هناك؟ قولي 'yes' أو 'ماشي'."
        else:
            reply = f"You can find that at {store_name} (Room {explicit_dest}). Want me to take you there? Say 'yes' or 'ok'."
        history.append((user_text, reply))
        try: gTTS(reply, lang="ar" if lang=="ar" else "en").save("/content/voice.mp3")
        except: pass
        return history, None, "/content/voice.mp3", session

    # --- 6. NAVIGATION ---
    if wants_nav and explicit_dest is not None:
        dest_room = explicit_dest
        store_name = ROOM_INFO.get(dest_room, f"Room {dest_room}")

        # Resolve START point (Priority order)
        # 1. User explicitly said start room in chat
        # 2. Tab 2 location (ALWAYS use this as default)
        # 3. Chat memory (fallback only)
        start_floor, start_xy = None, None

        if explicit_start is not None:
            start_floor = get_floor(explicit_start)
            start_xy = get_centroid(explicit_start)

        if start_xy is None:
            start_floor = session.get("start_floor")
            start_xy = session.get("start_xy")

        if start_xy is None:
            start_floor = session.get("chat_start_floor")
            start_xy = session.get("chat_start_xy")

        if start_floor is None or start_xy is None:
            msg = ("Please set your starting location in the Shop & Navigate tab first, or tell me where you are (e.g., 'from room 350')."
                   if lang != "ar" else "يرجى تحديد موقعك أولاً في تبويب التسوق والتنقل، أو أخبرني من أين تبدأ (مثال: من غرفة 350).")
            history.append((user_text, msg))
            try: gTTS(msg, lang="ar" if lang=="ar" else "en").save("/content/voice.mp3")
            except: pass
            return history, None, "/content/voice.mp3", session

        # Determine navigation mode based on special_needs
        nav_mode = session.get("nav_mode", "Normal")
        if session.get("special_needs"):
            nav_mode = "Special Needs"

        # Execute navigation
        img_path, instructions = navigate(start_floor, start_xy, dest_room, nav_mode)

        # Save history
        save_navigation(session.get("username", "guest"), (start_floor, start_xy), dest_room)
        save_navigation_mem(f"floor={start_floor},x={start_xy[0]},y={start_xy[1]}", dest_room)

        # Update chat memory: user is now at destination
        dest_centroid = get_centroid(dest_room)
        if dest_centroid:
            session["chat_start_floor"] = get_floor(dest_room)
            session["chat_start_xy"] = dest_centroid
        session["chat_dest_room"] = dest_room
        session["last_dest_room"] = dest_room
        session["last_referenced_room"] = dest_room

        # Build response
        if lang == "ar":
            reply = f"🗺️ تمام، هنروح {store_name} (غرفة {dest_room})\n\n{instructions}"
            voice = f"تمام، هنروح {store_name}. {instructions.replace(chr(10), ' ')}"
        else:
            reply = f"🗺️ Navigating to {store_name} (Room {dest_room})\n\n{instructions}"
            voice = f"Navigating to {store_name}. {instructions.replace(chr(10), ' ')}"

        history.append((user_text, reply))
        try: gTTS(voice, lang="ar" if lang=="ar" else "en").save("/content/voice.mp3")
        except: pass
        return history, img_path, "/content/voice.mp3", session

    # --- 7. FALLBACK: Try local reply or simple response ---
    local = local_chat_reply(user_text)
    if local:
        # If local reply contains store info, try to extract room for memory
        room = find_room_in_text(user_text)
        if room:
            session["last_referenced_room"] = room
            session["chat_dest_room"] = room
        history.append((user_text, local))
        try: gTTS(local).save("/content/voice.mp3")
        except: pass
        return history, None, "/content/voice.mp3", session

    # Simple fallback with optional local LLM
    if lang == "ar":
        reply = "مش فاهم قصدك. قولي اسم المتجر، المنتج، أو رقم الغرفة."
    else:
        reply = "I'm not sure what you mean. Try mentioning a store name, product, or room number like 351."

    # Try local LLM for a nicer fallback
    llm_reply = _generate_llm_reply(user_text, lang)
    if llm_reply:
        reply = llm_reply

    history.append((user_text, reply))
    try: gTTS(reply, lang="ar" if lang=="ar" else "en").save("/content/voice.mp3")
    except: pass
    return history, None, "/content/voice.mp3", session


def speech_to_text(audio):
    if audio is None:
        return ""
    try:
        sound = AudioSegment.from_file(audio)
        wav = "/content/temp.wav"
        sound.export(wav, format="wav")
        r = sr.Recognizer()
        with sr.AudioFile(wav) as src:
            data = r.record(src)
        # Try Arabic first, then English
        try: return r.recognize_google(data, language="ar-SA")
        except: return r.recognize_google(data, language="en-US")
    except Exception:
        return ""

# ============================================================
# UI HELPERS
# ============================================================
def format_products_html(promos, store_name, room):
    if not promos:
        return "<div style='color:#ff6b6b;padding:20px;text-align:center;font-family:monospace'>❌ No products found in your budget range.</div>"

    cards = ""
    for p in promos:
        stars = "⭐" * int(round(float(p.get("Rating", 0))))
        cards += f"""
        <div style="background:linear-gradient(135deg,#1a1a2e,#16213e);border:1px solid #00d4ff33;
                    border-radius:12px;padding:16px;margin-bottom:12px;">
          <div style="display:flex;justify-content:space-between;align-items:flex-start;gap:10px;">
            <div style="flex:1;">
              <div style="font-size:11px;color:#888;font-family:monospace;margin-bottom:4px">{p.get('Brand','N/A')}</div>
              <div style="color:#e0e0e0;font-size:13px;font-family:monospace;line-height:1.4">{p.get('Name','Unknown')}</div>
              <div style="margin-top:8px;font-size:12px;color:#ffd700">{stars} {p.get('Rating',0)} ({int(p.get('Reviews',0)):,} reviews)</div>
            </div>
            <div style="text-align:right;flex-shrink:0;">
              <div style="color:#ff6b6b;font-size:12px;text-decoration:line-through;font-family:monospace">{p.get('Original','0')} EGP</div>
              <div style="color:#00ff88;font-size:16px;font-weight:bold;font-family:monospace">{p.get('Discounted','0')} EGP</div>
              <div style="background:#ff4500;color:white;border-radius:20px;padding:3px 10px;
                          font-size:11px;font-weight:bold;margin-top:4px;font-family:monospace">{p.get('Discount',0)}% OFF 🔥</div>
            </div>
          </div>
        </div>"""

    return f"""
    <div style="font-family:monospace;max-height:500px;overflow-y:auto;padding:4px;">
      <div style="color:#00d4ff;font-size:14px;font-weight:bold;margin-bottom:12px;
                  padding-bottom:8px;border-bottom:1px solid #00d4ff33;">
        🏪 {store_name} · Room {room}
      </div>
      {cards}
    </div>"""


def set_location_ui(x, y, floor_choice, session):
    session = set_session_defaults(session)
    if x is None or y is None:
        return "<div style='color:#ff6b6b;font-family:monospace'>❌ Please enter coordinates.</div>", session
    session["start_floor"] = 0 if str(floor_choice) == "3" else 1
    session["start_xy"] = (float(x), float(y))
    session["start_room"] = session["start_xy"]
    return f"<div style='color:#00ff88;font-family:monospace'>✅ Location saved: floor {int(floor_choice)}, x={x}, y={y}</div>", session


def recommend_store_for_session(session):
    session = set_session_defaults(session)
    if not session.get("username"):
        return "<div style='color:#ff6b6b;font-family:monospace'>❌ Please login first.</div>", session
    if session.get("start_xy") is None or session.get("start_floor") is None:
        return "<div style='color:#ff6b6b;font-family:monospace'>❌ Please enter your current location first.</div>", session

    if session.get("is_new", True):
        room = nearest_store(session["start_xy"], session["start_floor"])
        msg = "🆕 New user → Nearest store recommended"
    else:
        room = most_visited_store(session["username"])
        if room is None:
            room = nearest_store(session["start_xy"], session["start_floor"])
        msg = "🔁 Returning user → Most visited store recommended"

    store_name = ROOM_INFO.get(room, f"Room {room}")
    session["recommended_room"] = room
    session["nav_target_room"] = room
    session["last_referenced_room"] = room
    session["last_dest_room"] = room

    html = f"""
    <div style="background:linear-gradient(135deg,#1a1a2e,#0d2137);border:1px solid #00d4ff55;
                border-radius:12px;padding:20px;font-family:monospace;">
      <div style="color:#888;font-size:11px;letter-spacing:2px">{msg}</div>
      <div style="color:#00d4ff;font-size:22px;font-weight:bold;margin:8px 0">{store_name}</div>
      <div style="color:#555;font-size:12px">Room {room}</div>
    </div>"""
    return html, session


def update_cat(type_name):
    return gr.update(choices=get_categories_for_type(type_name), value=None)


def update_sub(type_name, cat_name):
    return gr.update(choices=get_subcategories(type_name, cat_name), value=None)


def handle_agreement(choice, session):
    session = set_session_defaults(session)
    session["accepted_store"] = (choice or "").startswith("✅")
    types = [t["type"] for t in categories_data]
    if session["accepted_store"]:
        return (
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(choices=types, visible=True),
        )
    return (
        gr.update(choices=types, visible=True),
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(choices=types, visible=True),
    )


def do_navigate(choice, type_name, cat_name, sub_name, nav_mode_val, danger_room, session):
    session = set_session_defaults(session)

    if not session.get("username"):
        return "❌ Please login first.", None

    if session.get("start_xy") is None or session.get("start_floor") is None:
        return "❌ Please set your location first.", None

    reset_grids()

    # =========================================================
    # DESTINATION SELECTION
    # =========================================================
    if "Yes" in (choice or ""):
        dest_room = session.get("recommended_room")
        if dest_room is None:
            return "❌ Please get a recommendation first.", None
    else:
        if not sub_name:
            return "❌ Please select a sub-category.", None

        parent = SUB_TO_CATEGORY.get(sub_name, cat_name)
        store = STORE_CLUSTER.get(parent, parent)
        dest_room = STORE_CLUSTER_ROOM.get(store)

        if dest_room is None:
            return "❌ Could not map the selected category to a store.", None

    session["dest_room"] = dest_room

    # =========================================================
    # NAVIGATION MODE (FIXED with special_needs from session)
    # =========================================================
    if session.get("special_needs"):
        effective_mode = "Special Needs"
    else:
        effective_mode = nav_mode_val

    session["nav_mode"] = effective_mode

    # =========================================================
    # DANGER / CROWD HANDLING
    # =========================================================
    room_val = None
    try:
        if danger_room is not None and str(danger_room).strip() != "":
            room_val = int(float(danger_room))
    except Exception:
        room_val = None

    global fire_active, fire_room, fire_step, fire_center
    global crowded_active, crowded_room, crowded_center

    fire_active = False
    crowded_active = False
    fire_room = None
    crowded_room = None
    fire_center = None
    crowded_center = None

    if effective_mode == "Fire" and room_val is not None and room_val in ROOM_INFO:
        fire_active = True
        fire_room = room_val
        fire_center = get_centroid(room_val)

        if fire_center is not None:
            fire_step += 1
            radius = 20 + fire_step * 6

            if get_floor(room_val) == 0:
                mark_danger(grid_0, fire_center, radius)
            else:
                mark_danger(grid_1, fire_center, radius)

    if effective_mode == "Crowded" and room_val is not None and room_val in ROOM_INFO:
        crowded_active = True
        crowded_room = room_val
        crowded_center = get_centroid(room_val)

        if crowded_center is not None:
            if get_floor(room_val) == 0:
                mark_crowd(grid_0, crowded_center)
            else:
                mark_crowd(grid_1, crowded_center)

    # =========================================================
    # SAVE HISTORY
    # =========================================================
    save_navigation(
        session["username"],
        (session["start_floor"], session["start_xy"]),
        dest_room
    )

    save_navigation_mem(
        f"floor={session['start_floor']},x={session['start_xy'][0]},y={session['start_xy'][1]}",
        dest_room
    )

    # =========================================================
    # NAVIGATION CALL
    # =========================================================
    img_path, instructions = navigate(
        session["start_floor"],
        session["start_xy"],
        dest_room,
        effective_mode
    )

    return instructions, img_path


def do_get_products(type_name, cat_name, sub_name, session):
    session = set_session_defaults(session)
    if not session.get("username"):
        return "<div style='color:#ff6b6b;font-family:monospace'>❌ Please login first.</div>"
    if not sub_name:
        return "<div style='color:#ff6b6b;font-family:monospace'>❌ Please select a sub-category.</div>"

    if type_name and cat_name and sub_name:
        save_preference(session["username"], (type_name, cat_name, sub_name))

    gender = session.get("gender", "other")
    low = session.get("low", 500)
    high = session.get("high", 5000)
    result = recommend_for_user(sub_name, gender, low, high)
    session["selected_subcategory"] = sub_name
    return format_products_html(result["products"], result["store"], result["room"])

# ============================================================
# MAIN GRADIO APP
# ============================================================
with gr.Blocks(
    theme=gr.themes.Base(primary_hue="cyan", secondary_hue="blue", neutral_hue="slate"),
    css="""
    body { background: #0a0a14 !important; }
    .gradio-container { background: #0a0a14 !important; max-width: 1200px !important; }
    .tab-nav button { font-family: monospace; letter-spacing: 2px; font-size: 12px; }
    .panel { background: #111827; border: 1px solid #1e3a5f; border-radius: 12px; padding: 20px; }
    h1, h2, h3 { font-family: monospace !important; letter-spacing: 3px !important; }
    footer { display: none !important; }
    """,
    title="SmartMall AI System",
) as demo:

    session_state = gr.State({})

    gr.HTML("""
    <div style="text-align:center;padding:24px 0 8px;font-family:monospace;">
      <div style="font-size:28px;font-weight:900;letter-spacing:6px;
                  background:linear-gradient(90deg,#00d4ff,#00ff88,#ff4466);
                  -webkit-background-clip:text;-webkit-text-fill-color:transparent">
        SMART MALL AI
      </div>
      <div style="color:#555;font-size:11px;letter-spacing:4px;margin-top:6px">
        NAVIGATION · RECOMMENDATIONS · ASSISTANT
      </div>
    </div>
    """)

    with gr.Tabs():
        # ----------------------------------------------------
        # TAB 1 — AUTH
        # ----------------------------------------------------
        with gr.Tab("🔐 LOGIN / SIGNUP"):
            gr.HTML("<div style='color:#00d4ff;font-family:monospace;font-size:13px;margin-bottom:16px'>Enter your credentials to begin</div>")
            with gr.Row():
                with gr.Column(scale=1):
                    auth_username = gr.Textbox(label="Username", placeholder="your_username")
                    auth_password = gr.Textbox(label="Password", type="password", placeholder="••••••••")
                    auth_name = gr.Textbox(label="Full Name (signup only)", placeholder="John Doe")
                    auth_age = gr.Number(label="Age (signup only)", value=25)
                    auth_gender = gr.Dropdown(["male", "female", "other"], label="Gender (signup only)", value="male")
                    special_needs = gr.Radio(
                    ["No", "Yes"],
                    label="Are you a person with special needs?",
                    value="No"
                )
                    auth_budget_l = gr.Number(label="Min Budget EGP (signup only)", value=500)
                    auth_budget_h = gr.Number(label="Max Budget EGP (signup only)", value=5000)

                    with gr.Row():
                        signup_btn = gr.Button("SIGNUP", variant="primary")
                        login_btn = gr.Button("LOGIN", variant="secondary")
                    auth_status = gr.HTML()

            def do_signup(username, password, name, age, gender, low, high, special, session):
                cursor.execute("SELECT username FROM users WHERE username=?", (username,))
                if cursor.fetchone():
                    return "<div style='color:#ff6b6b;font-family:monospace'>❌ Username already exists.</div>", session
                # FIXED: Save special_needs to database as INTEGER
                special_flag = 1 if special == "Yes" else 0
                cursor.execute(
                    "INSERT INTO users VALUES (?,?,?,?,?,?,?,?)",
                    (username, password, name, int(age), gender, low, high, special_flag),
                )
                conn.commit()
                session = set_session_defaults(session)
                session.update({
                    "username": username, "gender": gender, "low": low, "high": high, "is_new": True,
                    "special_needs": (special == "Yes")
                })
                session["nav_mode"] = "Special Needs" if special == "Yes" else "Normal"
                return f"<div style='color:#00ff88;font-family:monospace'>✅ Welcome, {name}! Account created.</div>", session

            def do_login(username, password, session):
                cursor.execute("SELECT * FROM users WHERE username=? AND password=?", (username, password))
                user = cursor.fetchone()
                if not user:
                    return "<div style='color:#ff6b6b;font-family:monospace'>❌ Invalid credentials.</div>", session
                session = set_session_defaults(session)
                session.update({"username": user[0], "gender": user[4], "low": user[5], "high": user[6], "is_new": False})
                return f"<div style='color:#00ff88;font-family:monospace'>✅ Welcome back, {user[2]}!</div>", session

            signup_btn.click(do_signup, [auth_username, auth_password, auth_name, auth_age, auth_gender, auth_budget_l, auth_budget_h, special_needs, session_state], [auth_status, session_state])
            login_btn.click(do_login, [auth_username, auth_password, session_state], [auth_status, session_state])

        with gr.Tab("🏪 SHOP & NAVIGATE"):
            gr.HTML("<div style='color:#00d4ff;font-family:monospace;font-size:13px;margin-bottom:16px'>Get store recommendations and navigate</div>")
            gr.HTML("<div style='color:#888;font-family:monospace;font-size:11px;margin-bottom:8px'>STEP 0 — SET YOUR LOCATION</div>")
            with gr.Row():
                loc_x = gr.Number(label="X Coordinate")
                loc_y = gr.Number(label="Y Coordinate")
                loc_floor = gr.Dropdown(["3", "4"], label="Floor", value="3")
                set_loc_btn = gr.Button("📍 Set Location")
            loc_status = gr.HTML()

            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML("<div style='color:#888;font-family:monospace;font-size:11px;margin-bottom:8px'>STEP 1 — GET RECOMMENDATION</div>")
                    recommend_btn = gr.Button("🎯 Get Store Recommendation", variant="primary")
                    rec_result_html = gr.HTML()
                    rec_agree = gr.Radio(["✅ Yes, navigate me there", "❌ No, I'll choose my own"], label="Accept recommendation?")

                    gr.HTML("<div style='color:#888;font-family:monospace;font-size:11px;margin-top:20px;margin-bottom:8px'>STEP 2 — CHOOSE CATEGORY</div>")
                    cat_type = gr.Dropdown([t["type"] for t in categories_data], label="Main Category")
                    cat_cat = gr.Dropdown([], label="Category")
                    cat_sub = gr.Dropdown([], label="Sub-Category")
                    nav_mode = gr.Dropdown(["Normal", "Fire", "Crowded", "Special Needs"], label="Navigation Mode", value="Normal")
                    danger_room = gr.Textbox(label="Fire/Crowded Room Number", placeholder="Example: 350")
                    navigate_btn = gr.Button("🗺️ Start Navigation", variant="primary")

                with gr.Column(scale=1):
                    nav_instructions = gr.Textbox(label="Navigation Instructions", lines=10, interactive=False)
                    nav_image = gr.Image(label="Mall Map", type="filepath")

            gr.HTML("<div style='color:#888;font-family:monospace;font-size:11px;margin-top:20px;margin-bottom:8px'>STEP 3 — PRODUCT RECOMMENDATIONS</div>")
            with gr.Row():
                prod_type = gr.Dropdown([], label="Category Type")
                prod_cat = gr.Dropdown([], label="Category")
                prod_sub = gr.Dropdown([], label="Sub-Category")
            get_products_btn = gr.Button("🛍️ Get Product Recommendations", variant="primary")
            products_html = gr.HTML()

            set_loc_btn.click(set_location_ui, [loc_x, loc_y, loc_floor, session_state], [loc_status, session_state])
            recommend_btn.click(recommend_store_for_session, [session_state], [rec_result_html, session_state])
            rec_agree.change(handle_agreement, [rec_agree, session_state], [cat_type, cat_cat, cat_sub, prod_type])
            cat_type.change(update_cat, [cat_type], [cat_cat])
            cat_cat.change(update_sub, [cat_type, cat_cat], [cat_sub])
            prod_type.change(update_cat, [prod_type], [prod_cat])
            prod_cat.change(update_sub, [prod_type, prod_cat], [prod_sub])

            navigate_btn.click(do_navigate, [rec_agree, cat_type, cat_cat, cat_sub, nav_mode, danger_room, session_state], [nav_instructions, nav_image])
            get_products_btn.click(do_get_products, [prod_type, prod_cat, prod_sub, session_state], [products_html])

            def init_dropdowns():
                types = [t["type"] for t in categories_data]
                return gr.update(choices=types), gr.update(choices=types)
            demo.load(init_dropdowns, [], [prod_type, cat_type])

        with gr.Tab("🤖 AI ASSISTANT"):
            gr.HTML("<div style='color:#00d4ff;font-family:monospace;font-size:13px;margin-bottom:16px'>Chat with the mall AI assistant (English, Arabic, Egyptian slang)</div>")
            chatbot_widget = gr.Chatbot(
                label="SmartMall Assistant", height=480,
                value=[("", "👋 Welcome to SmartMall! I understand English, Arabic, and Egyptian slang. Ask me about any store or product, or say 'take me to room 351' / 'عايز اروح غرفة 351'.")],
            )
            with gr.Row():
                chat_msg = gr.Textbox(placeholder="Ask me anything about the mall...", label="Message", scale=4)
                chat_mic = gr.Audio(source="microphone", type="filepath", label="🎤", scale=1)
            chat_audio_out = gr.Audio(label="Voice Response")
            chat_img_out = gr.Image(label="Navigation Map", type="filepath")

            def chat_respond(msg, history, session):
                if not msg.strip(): return history, None, None, session
                history, img, audio, session = chatbot_respond(msg, history, session)
                return history, img, audio, session

            def voice_chat(audio, history, session):
                text = speech_to_text(audio)
                if not text:
                    history.append(("🎤 [voice]", "Sorry, I couldn't understand the audio."))
                    return history, None, None, session
                return chat_respond(text, history, session)

            chat_msg.submit(chat_respond, [chat_msg, chatbot_widget, session_state], [chatbot_widget, chat_img_out, chat_audio_out, session_state])
            chat_mic.change(voice_chat, [chat_mic, chatbot_widget, session_state], [chatbot_widget, chat_img_out, chat_audio_out, session_state])

if __name__ == "__main__":
    demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
IMPORTANT: You are using gradio version 3.39.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://5bdec25e7907d2b1ca.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5bdec25e7907d2b1ca.gradio.live
